# Test API — Async (202 + polling)
POST returns immediately with 202. Poll `/results/<load_id>` until done.


## Cell 1 — Health check

In [ ]:
import requests, json, time

BASE_URL = 'https://desrapidev-fqaphqeaeuc7hafh.westcentralus-01.azurewebsites.net'
r = requests.get(f'{BASE_URL}/', timeout=10)
print(r.status_code, r.json())


## Cell 2 — POST payload (returns immediately with 202)

In [ ]:
import math
from pathlib import Path

raw = json.loads(Path('./payloads/medicare_input_loadid142.json').read_text(encoding='utf-8'))
rows = raw['pbp'] if isinstance(raw, dict) and 'pbp' in raw else raw

def sanitize(obj):
    if isinstance(obj, dict):  return {k: sanitize(v) for k, v in obj.items()}
    if isinstance(obj, list):  return [sanitize(v) for v in obj]
    if isinstance(obj, float) and (math.isnan(obj) or math.isinf(obj)): return None
    return obj

print(f'Sending {len(rows):,} rows...')
r = requests.post(f'{BASE_URL}/save', json=sanitize(rows),
                  headers={'Content-Type': 'application/json'}, timeout=30)

print(f'HTTP Status: {r.status_code}')  # Should be 202
resp = r.json()
print(json.dumps(resp, indent=2))

load_id   = resp['load_id']
blob_name = resp['blob_name']
print(f'\nload_id:   {load_id}')
print(f'blob_name: {blob_name}')
print('Now run Cell 3 to poll for results.')


## Cell 3 — Poll /results until done
Polls every 15 seconds until status is `success` or `error`.


In [ ]:
import pandas as pd

POLL_INTERVAL = 15   # seconds between polls
MAX_WAIT      = 600  # stop after 10 minutes

print(f'Polling /results/{load_id} every {POLL_INTERVAL}s...')
elapsed = 0
while elapsed < MAX_WAIT:
    r    = requests.get(f'{BASE_URL}/results/{load_id}', timeout=30)
    data = r.json()
    status = data.get('status')

    print(f'  [{elapsed:>3}s] status={status}')

    if status == 'success':
        print(f'\nDone! {data["result_count"]} rows in blob: {data["blob"]}')
        df = pd.DataFrame(data['results'])
        display(df[['benefitid','benefitname','serviceTypeDesc','tinyDescription']])
        break

    if status == 'error':
        print(f'\nERROR: {data.get("error")}')
        break

    time.sleep(POLL_INTERVAL)
    elapsed += POLL_INTERVAL
else:
    print('Timed out waiting for results.')


## Cell 4 — Save CSV

In [ ]:
CSV_OUT = f'output_benefits_loadid_{load_id}.csv'
df.to_csv(CSV_OUT, index=False)
print(f'Saved {len(df)} rows to {CSV_OUT}')
